# The debt extension — reconciling the register to the official totals

This notebook explains the five `debt_*.csv` files in
[`../deliverables/`](../deliverables): the reconciliation of central-government
debt securities interest and financing to the package's own general-government
aggregates (`GF01_7`, `NLB`), for the United Kingdom, France and Germany.

**Read this before the charts below.** The per-security register described in
`DEBT_KICKOFF.md` — every gilt, OAT and Bund, its coupon, its position through
time — cannot yet be built: the debt-office hosts (DMO, AFT, Finanzagentur)
are unreachable in this run (OQ-8). What this notebook shows instead is the
**aggregate class-level layer** (DD8): the ministries' and statistical
offices' own published totals *by instrument class* (`debt_class_aggregates`),
standing in for the sum a per-security register would otherwise produce. The
two reconciliation chains — interest to `GF01_7`, financing to `NLB` — are
built on top of that aggregate layer exactly as `DEBT_KICKOFF.md` §8
specifies: register sum → step A (finance-ministry cash) → step B (S.1311
national-accounts) → step C (S.13 package total), with every bridge item
either official or declared, and

    official_total = carried_from_previous_step + Σ bridge items + residual

held per (country, year, step). **Every residual here is published, never
allocated** — no line in this notebook is scaled, tilted or adjusted to make
a total match; where the chain does not close, the gap is drawn as its own
bar or its own line.

Everything below is read from `../deliverables/` and nothing is recomputed
from `../data/`.


In [1]:
import io
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image as Image_display, display
from PIL import Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "deliverables" / "debt_reference_series.csv").exists())
DELIV = ROOT / "deliverables"
load = lambda n: pd.read_csv(DELIV / f"{n}.csv", float_precision="round_trip")

REF = load("debt_reference_series")
OFFICIAL = load("debt_official_totals")
CLASS_AGG = load("debt_class_aggregates")
INTEREST = load("debt_interest_reconciliation")
FINANCING = load("debt_financing_reconciliation")

REF["date"] = pd.to_datetime(REF["date"])
COUNTRY = {"GBR": "United Kingdom", "FRA": "France", "DEU": "Germany"}

# Fixed categorical palette, assigned by role and reused across every chart in
# this notebook: the same colour always means the same instrument class or
# the same chain step, whichever country or section it appears in.
BLUE, ORANGE, GREEN, PURPLE, RED, TEAL, BROWN, GOLD = (
    "#2a78d6", "#eb6834", "#1baf7a", "#8456ce",
    "#d64550", "#2ba8a0", "#a8763e", "#d1a62b",
)
INK, MUTED, AXIS = "#0b0b0b", "#898781", "#c3c2b7"
SURFACE, SHADE = "#fcfcfb", "#eceae4"

CLASS_COLOR = {
    "fixed_bullet": BLUE, "floating": PURPLE, "inflation_linked": GREEN,
    "bill": ORANGE, "other": BROWN, "mixed": TEAL,
}
STEP_LABEL = {
    "A_cg_cash": "A · CG cash interest", "B_s1311_d41": "B · S.1311 D.41",
    "C_s13_gf01_7": "C · GF01_7",
    "A_cg_cash_requirement": "A · CG net cash requirement",
    "B_s1311_b9": "B · S.1311 B.9", "C_s13_nlb": "C · NLB",
}
STEP_COLOR = {"A": BLUE, "B": ORANGE, "C": GREEN}


def step_color(step):
    return STEP_COLOR[step[0]]


plt.rcParams.update({
    "figure.figsize": (7, 3), "figure.dpi": 72,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "grid.color": AXIS, "grid.alpha": 0.45, "grid.linewidth": 0.6,
    "legend.frameon": False, "legend.fontsize": 8.5,
    "lines.linewidth": 1.8,
})


def _palette():
    '''A fixed palette of exactly the colours these charts use.

    Quantising a PNG at full colour depth is what keeps this notebook under
    1 MB and renderable everywhere; an *adaptive* palette would instead
    allocate slots by pixel count and crush thin lines to a nearby background
    tint. Fixing the palette -- each mark colour plus its blends toward the
    two backgrounds -- means every hue survives regardless of how little of
    the canvas it covers.
    '''
    from matplotlib.colors import to_rgb

    entries = [to_rgb(SURFACE), to_rgb(SHADE)]
    marks = (INK, MUTED, AXIS, BLUE, ORANGE, GREEN, PURPLE, RED, TEAL, BROWN, GOLD)
    for mark in marks:
        m = to_rgb(mark)
        for ground in (SURFACE, SHADE):
            g = to_rgb(ground)
            entries += [tuple(m[i] + (g[i] - m[i]) * t for i in range(3))
                        for t in (0.0, 0.2, 0.4, 0.6, 0.8)]
    flat = [round(255 * v) for e in entries for v in e]
    pal = Image.new("P", (1, 1))
    pal.putpalette(flat + [0] * (768 - len(flat)))
    return pal


PALETTE = _palette()


def show(fig):
    '''Display a figure as a PNG quantised onto the fixed palette above.'''
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=plt.rcParams["figure.dpi"], bbox_inches="tight")
    plt.close(fig)
    small = Image.open(buf).convert("RGB").quantize(
        palette=PALETTE, dither=Image.Dither.NONE)
    out = io.BytesIO()
    small.save(out, format="PNG", optimize=True)
    display(Image_display(data=out.getvalue()))


def thousands(ax):
    ax.yaxis.set_major_formatter(
        plt.matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
